
# SQLite db management
> SQLite DB

In [ ]:
#| default_exp core.sqldb

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, logging, typing 
import os, csv, json, gzip, sqlite3
import subprocess, queue, threading

import pandas as pd

from typing import Any, NamedTuple
from contextlib import contextmanager
from edcompanion.core import configuration
from edcompanion.threadworkers import bind_queue_as_generator_blocking


In [ ]:
sqlite3.sqlite_version

'3.51.1'

In [ ]:
from confproxy.core import init_console_logging

init_console_logging(__name__)

2025-12-23T21:24:02+0100 INFO	17295	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
syslog.info(f"Loading module {__name__}, python version {sys.version}, sqlite version {sqlite3.sqlite_version}")


2025-12-23T21:24:02+0100 INFO	17295	__main__	3208081707.py	<module>	3	Loading module __main__, python version 3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0], sqlite version 3.51.1


## SQLite

### SQLiteQueryParams

In [ ]:
#| export
class SQLiteQueryParams(NamedTuple):
    as_param: typing.Callable
    append_param: typing.Callable   
    get_params: typing.Callable  


In [ ]:
#| export

def sqlite_query_params(log=None) -> SQLiteQueryParams:
    sql_params = {}

    def as_param(name:str):
        return f":{str(name)}"
   
    def append_param(name:str, value:Any):
        assert str(name) not in sql_params, f"Duplicate parameter {name}"
        assert len(sql_params) < 32766, "SQLite does not allow more then approx. 32k bound parameters"

        last_name = str(name)
        sql_params[last_name] = value
        return as_param(last_name)
    
    def get_params():
        return sql_params.copy()
    
    return SQLiteQueryParams(
        as_param=as_param,
        append_param=append_param if log is None else lambda p: log(append_param(p)),
        get_params=get_params
    )

### SQLiteConnectionInterface

In [ ]:
#| export

class SQLiteConnectionInterface(typing.NamedTuple):
    cursor: typing.Callable
    commit: typing.Callable
    rollback: typing.Callable
    close: typing.Callable
    execute: typing.Callable
    executemany: typing.Callable


In [ ]:
sys.version_info

sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0)

In [ ]:
#| export
def sqllite_connection_interface(
        database:str=":memory:",
    ) -> SQLiteConnectionInterface:

    vi = sys.version_info
    assert vi.major >= 3, f"Python >= 3.0 required. Found {vi.major}.{vi.minor}.{vi.micro}"

    if vi.minor > 11:   
        syslog.info("Using legacy autocommit")
        connection = sqlite3.connect(database, autocommit=sqlite3.LEGACY_TRANSACTION_CONTROL, isolation_level='DEFERRED')
    else:
        syslog.info("Using isolation_level=None")
        connection = sqlite3.connect(database, isolation_level=None)

    def close():
        syslog.info("Closing connection")
        connection.close()

    def commit():
        syslog.info("Committing on connection")
        connection.commit()

    def rollback():
        syslog.info("Rolling back connection")
        connection.rollback()
    
    def cursor():
        syslog.info("Creating cursor")
        return connection.cursor()
    
    def execute(sql:str, params:tuple|dict=()):
        return connection.execute(sql, params)
    
    def execute_many(sql:str, params:list[tuple|dict]=[]):
        return connection.executemany(sql, params)

    syslog.info("Returning connection-interface")
    return SQLiteConnectionInterface(
        cursor=cursor,
        commit=commit,
        rollback=rollback,
        close=close,
        execute=execute,
        executemany=execute_many
    )

### Context manager

In [ ]:
#| export

@contextmanager
def sqllite_connection(*args, **kwargs):
    syslog.info(f"Opening connection-interface to {args}")
    interface = sqllite_connection_interface(*args, **kwargs)
    try:
        yield interface
    finally:
        interface.close()

### Test

In [ ]:
with sqllite_connection() as ci:
    
    ci.execute("CREATE TABLE lang(name, first_appeared)")

    # This is the named style used with executemany():
    data = (
        {"name": "C", "year": 1972},
        {"name": "Fortran", "year": 1957},
        {"name": "Python", "year": 1991},
        {"name": "Go", "year": 2009},
    )
    ci.executemany("INSERT INTO lang VALUES(:name, :year)", data)
    for r in ci.execute("SELECT * FROM lang"):
        print(r)
        
    # This is the qmark style used in a SELECT query:
    params = (1972,)

    for r in ci.execute("SELECT * FROM lang WHERE first_appeared = :year", {'year':1957}):
        print(r)



2025-12-23T21:24:02+0100 INFO	17295	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ()
2025-12-23T21:24:02+0100 INFO	17295	__main__	3552464633.py	sqllite_connection_interface	10	Using legacy autocommit
2025-12-23T21:24:02+0100 INFO	17295	__main__	3552464633.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-23T21:24:02+0100 INFO	17295	__main__	3552464633.py	close	17	Closing connection


('C', 1972)
('Fortran', 1957)
('Python', 1991)
('Go', 2009)
('Fortran', 1957)


## Table definition

In [ ]:
class DBIndexDefinition(NamedTuple):
    unique: bool
    index_type: str

class DBColumnDefinition(NamedTuple):
    name: str
    data_type: str
    index: DBIndexDefinition

class DBTableDefinition(NamedTuple):
    name: str
    columns: list[DBColumnDefinition]

def create_table(connection, table_def:DBTableDefinition):

    qs = "CREATE TABLE IF NOT EXISTS " + table_def.name + "(" + ",".join([f"{D.name} {D.data_type} " for D in table_def.columns]) + ")"
    syslog.info(qs)
    connection.execute(qs)

    for column in table_def.columns:
        if column.index:
            qs = f"CREATE {'UNIQUE' if column.index.unique else ''} INDEX  {table_def.name}_{column.name}_idx ON {table_def.name} ({column.name})" # "CREATE INDEX " + table_def.name + "_"  + column.name + "_idx ON " + table_def.name
            syslog.info(qs)
            connection.execute(qs)

    connection.commit()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()